# 01 · Persistent batch extraction

Run the expensive extraction step once. It reads recordings selected by `file_selection_policy: prefer_d5`, creates epochs only within strictly continuous runs, and groups context with a 90-second tolerance. It computes features, GSSC, YASA, and SleepFM outputs without storing signals or epochs.

## i) Imports

In [ ]:
from pathlib import Path

from IPython.display import display

from dmt_hypnodensities import (
    assemble_outputs, finalize_run, load_config, prepare_run, run_batch, save_table
)

## ii) Parameters and configuration

In [ ]:
CONFIG_PATH = next(path.resolve() for path in (Path('configs/analysis.yaml'), Path('../configs/analysis.yaml')) if path.is_file())
RUN_NAME = 'main_hybrid90_gssc_yasa_sleepfm_cpu_v1'

config = load_config(CONFIG_PATH)
assert config.gap_tolerance_seconds == 90.0
assert set(config.stagers) == {'gssc', 'yasa', 'sleepfm'}
assert config.file_selection_policy == 'prefer_d5'
assert config.compute_device == 'cpu'
run = prepare_run(config, RUN_NAME, reuse_existing=True)
display({
    'run': str(run.root),
    'raw_data': str(run.config.raw_dir),
    'stagers': run.config.stagers,
    'features': run.config.feature_analyses,
    'gap_tolerance_seconds': run.config.gap_tolerance_seconds,
    'compute_device': run.config.compute_device,
})

## iii) Main batched analysis

This is the only expensive cell. When a compatible run is reopened, `run_batch` reuses complete persisted recordings and retries only failed or incomplete recordings.

In [ ]:
batch_summary = run_batch(run.config, reuse_completed=True)
finalize_run(run, batch_summary)
display(batch_summary)

## iv) Control de calidad

In [ ]:
tables = assemble_outputs(run.recordings, strict=True)
staging_qc_summary = (
    tables.staging_qc.groupby(['stager', 'status'], dropna=False)
    .size().rename('n').reset_index()
)
block_qc_summary = tables.blocks.groupby(['condition'], dropna=False).agg(
    recordings=('recording_id', 'nunique'),
    blocks=('block_id', 'nunique'),
    epochs=('n_epochs', 'sum'),
).reset_index()
save_table(staging_qc_summary, run.tables / 'staging_qc_summary.csv')
save_table(block_qc_summary, run.tables / 'block_qc_summary.csv')
display(block_qc_summary, staging_qc_summary)

## v) Output review

Scientific figures are generated from the tables persisted by notebooks 02 and 03. This section checks coverage and row counts without mixing quality control with inference.

In [ ]:
display({
    'selected_recordings': len(tables.file_selection),
    'successful_recordings': int(tables.batch_summary['status'].eq('ok').sum()),
    'blocks': len(tables.blocks),
    'feature_rows': len(tables.features),
    'hypnodensity_rows': len(tables.hypnodensities),
    'spectrum_rows': len(tables.spectra),
})